In [1]:
from functools import partial
from pathlib import Path
from itertools import product

import colormaps
import dask.array as darr
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import xarray as xr
from dask.array.lib.stride_tricks import sliding_window_view
from jetutils.data import extract, open_da, standardize, standardize_polars_dtypes, smooth
from jetutils.definitions import (
    DATADIR,
    FIGURES,
    MONTH_NAMES,
    N_WORKERS,
    RADIUS,
    RESULTS,
    YEARS,
    Timer,
    compute,
    do_rle,
    explode_rle,
    get_index_columns,
    get_region,
    polars_to_xarray,
    squarify,
    xarray_to_polars,
    iterate_over_year_maybe_member,

)
from jetutils.derived_quantities import (
    compute_2d_div,
    compute_norm_derivative,
    convolve_dask,
    find_axis,
)
from jetutils.geospatial import (
    compute_alignment,
    create_bias_correction,
    create_jet_relative_dataset,
    detect_contours,
    detect_contours_lonlat,
    diff_exp,
    gather_normal_da_jets,
    haversine,
    interp_jets_to_zero_one,
    sort_by_index_then_difflon,
    sort_by_index_then_newindex,
    detect_overturnings,
    detect_streamers,
    sjoin_to_grid,
    to_xarray_sjoin,
    event_props,
)
from jetutils.jet_finding import (
    average_jet_categories,
    bias_correct,
    coarsen_da,
    connected_from_cross,
    do_everything,
    extract_features,
    gaussian_smooth_func,
    has_periodic_x,
    jet_integral_haversine,
    jet_position_as_da,
    persistence_expr,
    preprocess_ds,
    round_polars,
    to_one_large,
    track_jets,
    compute_widths,
    find_all_jets,
    pers_from_cross,
    is_polar_gmix,
    compute_jet_props
)
from jetutils.plots import (
    COLORS,
    COLORS_EXT,
    Clusterplot,
    make_transparent,
    num2tex,
    plot_dayofyear_trends,
    plot_seasonal,
)
from matplotlib.cm import ScalarMappable
from matplotlib.colors import (
    BoundaryNorm,
    LinearSegmentedColormap,
    hsv_to_rgb,
    rgb_to_hsv,
)
from matplotlib.ticker import MaxNLocator
from polars.exceptions import ColumnNotFoundError
from scipy.interpolate import splev, splprep
from scipy.ndimage import gaussian_filter
from tqdm import tqdm, trange

%load_ext IPython.extensions.autoreload
%autoreload 2
%matplotlib inline

no xgboost found
No shap


In [23]:
path = Path(DATADIR, "ERA5/plev/high_wind/6H/results/9")
ds = xr.open_dataset(path.joinpath("da.nc"))
ds = standardize(ds)
ds = extract(
    ds, minlon=-80, maxlon=40, minlat=15, maxlat=80
)
times = ds["time"].values

find_jets_kwargs = dict(
    n_coarsen=3,
    base_s_thresh=0.55,
    alignment_thresh=0.6,
    int_thresh_factor=0.6,
    hole_size=6,
    smooth_func=partial(gaussian_smooth_func, sigma_lon=2, sigma_lat=0.8),
)
jets, phat_jets, props, props_full = do_everything(ds, path, **find_jets_kwargs, track_large=False)

# stage 6: Interpolate new fields
args = ["all", None, -100, 60, 0, 88]

to_do = (
    ("t2m", "surf", "t2m", {}),
    ("tp", "surf", "tp", {}),
    ("PV330", "thetalev", "PV330", {}),
    ("PV350", "thetalev", "PV350", {}),
    ("APVO", "thetalev", "APVO", {}),
    ("CPVO", "thetalev", "CPVO", {}),
    ("SAPVS", "thetalev", "SAPVS", {}),
    ("TAPVS", "thetalev", "TAPVS", {}),
    ("SCPVS", "thetalev", "SCPVS", {}),
    ("TCPVS", "thetalev", "TCPVS", {}),
    ("theta", "surf", ("alot2pvu", "theta"), {}),
    ("EKE250", "plev", ("eddy_stuff", "EKE"), {}),
)

bias_correction = pl.read_parquet(path.joinpath("bias_correct.parquet"))

for huh in to_do:
    rename, levtype, name, kwargs = huh
    ofile = path.joinpath(f"{rename}_relative.parquet")
    if ofile.is_file():
        continue
    da = open_da("ERA5", levtype, name, "6H", *args, **kwargs).rename(rename)
    if rename in ["APVO", "CPVO", "SAPVS", "TAPVS", "SCPVS", "TCPVS"]:
        da = da.sel(lev=slice(340, 350)).any("lev")
    interpd = create_jet_relative_dataset(phat_jets, da, bias_correction=bias_correction, dn=1e5, n_interp=30)
    del da
    interpd.write_parquet(ofile)

 34%|███▍      | 22/64 [13:48<32:29, 46.42s/it]

In [ ]:
eddy_stuff = 1
for year in YEARS:
    da = xr.open_dataarray(Path(DATADIR, "ERA5", "thetalev", "PV330", "6H", f"{year}.nc"))
    da = standardize(da, do_chunk=True)
    da = da.sel(lat=slice(10, 80), lon=slice(-100, 60))
    da = da * 1e6
    da = smooth(da, {"lon": ("win", 7), "lat": ("win", 7)})
    da = compute(da, progress_flag=True)
    contours = detect_contours_lonlat(da, [2], 0, 8, do_round=True)
    sts = detect_streamers(contours, max_realdist=8e5, min_contourdist=1.5e6, max_contourdist=2e8, min_ratio=8)


[########################################] | 100% Completed | 16.27 s


In [ ]:
xr.open_dataarray(Path(DATADIR, "ERA5", "thetalev", "PV330", "6H", f"{year}.nc"))

total 2
-rwxr-xr-x 1 hb22g102 2045 Jun  2 09:48 jupyter-compute*
-rwxr-xr-x 1 hb22g102  477 Jun  1 12:05 jupyter_remote_port_forward*
-rw-r--r-- 1 hb22g102  231 Jun  2 14:40 launch_jupyter.sh


In [31]:
da_apvo_sev = xr.open_dataarray(Path(DATADIR, "ERA5", "thetalev", "APVO", "6H", "full.nc"))
da_apvo_sev = da_apvo_sev.sel(time=da_apvo_sev.time.dt.year==1959, lev=330)

In [32]:
da_apvo_sev

<xarray.DataArray 'flag' (time: 1460, lat: 101, lon: 161)> Size: 24MB
[23741060 values with dtype=int8]
Coordinates:
  * time     (time) datetime64[ns] 12kB 1959-01-01 ... 1959-12-31T18:00:00
  * lat      (lat) float64 808B 0.0 1.0 2.0 3.0 4.0 ... 97.0 98.0 99.0 100.0
  * lon      (lon) float64 1kB -100.0 -99.0 -98.0 -97.0 ... 57.0 58.0 59.0 60.0
    lev      int64 8B 330
Attributes:
    long_name:  flag wave breaking

# exp9

In [ ]:
path = Path(DATADIR, "ERA5/plev/high_wind/6H/results/9")
ds = xr.open_dataset(path.joinpath("da.nc"))
ds = extract(
    ds, minlon=-80, maxlon=40, minlat=15, maxlat=80
)
times = ds["time"].values
ds = standardize(ds)
find_jets_kwargs = dict(
    n_coarsen=3,
    base_s_thresh=0.55,
    alignment_thresh=0.6,
    int_thresh_factor=0.6,
    hole_size=6,
    smooth_func=partial(gaussian_smooth_func, sigma_lon=2, sigma_lat=0.8),
)
jets, phat_jets, props, props_full = do_everything(ds, path, **find_jets_kwargs)

In [ ]:
args = ["all", None, -100, 60, 0, 88]

to_do = (
    ("t2m", "surf", "t2m", {}),
    ("tp", "surf", "tp", {}),
    ("PV330", "thetalev", "PV330", {}),
    ("PV350", "thetalev", "PV350", {}),
    ("APVO", "thetalev", "APVO", {}),
    ("CPVO", "thetalev", "CPVO", {}),
    ("theta", "surf", ("alot2pvu", "theta"), {}),
    ("EKE250", "plev", ("eddy_stuff", "EKE"), {}),
)

bias_correction = pl.read_parquet(path.joinpath("bias_correct.parquet"))

for huh in to_do:
    rename, levtype, name, kwargs = huh
    ofile = path.joinpath(f"{rename}_relative.parquet")
    if ofile.is_file():
        continue
    da = open_da("ERA5", levtype, name, "6H", *args, **kwargs).rename(rename)
    if rename in ["APVO", "CPVO"]:
        da = da.sel(lev=slice(320, 350)).any("lev")
    interpd = create_jet_relative_dataset(
        phat_jets, da, bias_correction=bias_correction, dn=1e5, n_interp=30
    )
    del da
    interpd.write_parquet(ofile)

ds_eddies = xr.open_dataset(f"{DATADIR}/ERA5/plev/eddy_stuff/6H/full.zarr")
ds_eddies = ds_eddies.sel(lat=slice(None, 85))
to_do = {
    # "F1": ("F11", "F12"),
    # "F2": ("F12", "F22"),
    "hor": ("hor1", "hor2"),
}
for dest, sources in to_do.items():
    ofile = path.joinpath(f"{dest}_relative.parquet")
    if ofile.is_file():
        continue
    das = [ds_eddies[source] for source in sources]
    interpd = create_jet_relative_dataset(
        phat_jets,
        *das,
        bias_correction=bias_correction,
        align_2d=dest,
        dn=1e5,
        n_interp=30,
    )
    interpd.write_parquet(ofile)


In [ ]:
phat_props = props.filter(phat_filter)
index_columns = get_index_columns(phat_props)

phat_props = average_jet_categories(phat_props, polar_cutoff=0.5)
cross = cross.filter(pl.col("jet ID") == pl.col("jet ID_right"))
pers = cross.select("time", jet=pl.when(pl.col("jet ID") == 0).then(pl.lit("STJ")).otherwise(pl.lit("EDJ")), pers="pers", is_polar="is_polar")

In [ ]:
jets = to_one_large(jets, 1.3e8)
cross = standardize_polars_dtypes(pl.read_parquet(cross_path))
cross = pers_from_cross_catd(cross)

In [ ]:
ds = xr.open_dataset(
    f"{DATADIR}/ERA5/plev/high_wind/6H/results/8/da.nc"
)
ds = extract(
    ds, minlon=-80, maxlon=40, minlat=15, maxlat=80
)
times = ds["time"].values
path = Path(DATADIR, "ERA5/plev/high_wind/6H/results/9")

kwargs = dict(
    n_coarsen=3,
    base_s_thresh=0.6,
    alignment_thresh=0.8,
    int_thresh_factor=0.5,
    hole_size=6,
    smooth_func=partial(gaussian_smooth_func, sigma_lon=2, sigma_lat=0.8),
)
# jets, ph_jets, props, props_full = do_everything(ds, path, **kwargs)

In [ ]:
ds = standardize(ds)
save_path = path
jets_path = save_path.joinpath("jets.parquet")
props_path = save_path.joinpath("props.parquet")
props_final_path = save_path.joinpath("props_full.parquet")
cross_path = save_path.joinpath("cross.parquet")
bc_path = save_path.joinpath("bias_correct.parquet")

jets = standardize_polars_dtypes(pl.read_parquet(jets_path))
phat_jets = to_one_large(jets, int_EDJ_threshold=1.3e8)
if not bc_path.is_file():
    bc = create_bias_correction(phat_jets, ds)
    bc.write_parquet(bc_path)
# props = standardize_polars_dtypes(pl.read_parquet(props_path))
# props_catd = squarify(average_jet_categories(props), ["time", "jet"])
# cross = track_jets(jets, catd=False)
# cross_, summary_comp = connected_from_cross(jets, cross, dist_thresh=1.1e6, dis_polar_thresh=0.05)
# summary_comp = summary_comp.join(props, on=["time", "jet ID"], suffix="_huh")
# cross = cross.join(props, on=["time", "jet ID"], suffix="_huh")
# pers = cross.with_columns(pers=persistence_expr()).group_by("time", "jet ID", maintain_order=True).agg(pl.col("pers").max(), pl.col("is_polar_huh").first())
# props_catd = props_catd.join(
#     props_catd.rolling("time", period="2d", group_by="jet").agg(
#         **{
#             f"{col}_var": pl.col(col).var()
#             for col in ["mean_lon", "mean_lat", "mean_s", "s_star"]
#         }
#     ),
#     on=["time", "jet"],
# )
# phat_jets = to_one_large(jets, int_EDJ_threshold=1.3e8)
# phat_filter = (pl.col("is_polar") < 0.5) | ((pl.col("is_polar") > 0.5) & (pl.col("int") > 1.3e8))
# if not cross_path.is_file():
#     cross = track_jets(phat_jets, catd=True, yearly=True)
#     cross = pers_from_cross_catd(cross)
#     cross.write_parquet(cross_path)
# else:
#     cross = standardize_polars_dtypes(pl.read_parquet(cross_path))
#     cross = pers_from_cross_catd(cross)

In [ ]:
args = ["all", None, -100, 60, 0, 88]

to_do = (
    ("t2m", "surf", "t2m", {}),
    ("tp", "surf", "tp", {}),
    ("PV330", "thetalev", "PV330", {}),
    ("PV350", "thetalev", "PV350", {}),
    ("APVO", "thetalev", "APVO", {}),
    ("CPVO", "thetalev", "CPVO", {}),
    ("theta", "surf", ("alot2pvu", "theta"), {}),
    ("EKE250", "plev", ("eddy_stuff", "EKE"), {}),
)

bias_correction = pl.read_parquet(path.joinpath("bias_correct.parquet"))

for huh in to_do:
    rename, levtype, name, kwargs = huh
    ofile = path.joinpath(f"{rename}_relative.parquet")
    if ofile.is_file():
        continue
    da = open_da("ERA5", levtype, name, "6H", *args, **kwargs).rename(rename)
    if rename in ["APVO", "CPVO"]:
        da = da.sel(lev=slice(320, 350)).any("lev")
    interpd = create_jet_relative_dataset(phat_jets, da, bias_correction=bias_correction, dn=1e5, n_interp=30)
    del da
    interpd.write_parquet(ofile)

ds_eddies = xr.open_dataset(f"{DATADIR}/ERA5/plev/eddy_stuff/6H/full.zarr")
ds_eddies = ds_eddies.sel(lat=slice(None, 85))
to_do = {
    # "F1": ("F11", "F12"),
    # "F2": ("F12", "F22"),
    "hor": ("hor1", "hor2"),
}
for dest, sources in to_do.items():
    ofile = path.joinpath(f"{dest}_relative.parquet")
    if ofile.is_file():
        continue
    das = [ds_eddies[source] for source in sources]
    interpd = create_jet_relative_dataset(phat_jets, *das, bias_correction=bias_correction, align_2d=dest, dn=1e5, n_interp=30)
    interpd.write_parquet(ofile)

# EMF

In [ ]:
df_eddies = xr.open_dataset("/storage/workspaces/giub_meteo_impacts/ci01/ERA5/plev/eddy_stuff/6H/full.zarr")

In [ ]:
df_eddies["F12"][0].sel(lat=slice(None, 88)).plot()

In [ ]:
df_eddies["EKE"][0].sel(lat=slice(None, 88)).plot()

In [ ]:
df_eddies["F11"][0].sel(lat=slice(None, 88)).plot()

In [ ]:
df_eddies["hor1"][0].sel(lat=slice(None, 88)).plot()
df_eddies["EKE"][0].sel(lat=slice(None, 88)).plot.contour(levels=5, colors="black")

# jets = pl.read_parquet("/storage/workspaces/giub_meteo_impacts/ci01/ERA5/plev/high_wind/6H/results/10/jets.parquet")
# j = jets.filter(pl.col("time") == pl.col("time").unique().first())
# for _, j in j.group_by("jet ID"):
#     plt.plot(*j["lon", "lat"])

In [ ]:
from jetutils.jet_finding import pers_from_cross_catd

basepath = Path("/storage/workspaces/giub_meteo_impacts/ci01/ERA5/plev/high_wind/6H/results/10")
cross = pl.read_parquet(basepath.joinpath("cross.parquet"))
cross = cross.filter(pl.col("jet ID") == pl.col("jet ID_right"))
cross = pers_from_cross_catd(cross)
pers = cross.select("time", jet=pl.when(pl.col("jet ID") == 0).then(pl.lit("STJ")).otherwise(pl.lit("EDJ")), pers="pers")

In [ ]:
pers

In [ ]:
df_eddies["hor1"][:4000].mean("time").sel(lat=slice(None, 88)).plot()
df_eddies["EKE"][:4000].mean("time").sel(lat=slice(None, 88)).plot.contour(levels=5, colors="black")

# new jets bc

In [ ]:
path = Path(DATADIR, "ERA5/plev/high_wind/6H/results/10")
ds = xr.open_dataset(path.joinpath("da.nc"))
ds = standardize(ds)
ds = extract(
    ds, minlon=-80, maxlon=40, minlat=15, maxlat=80
)
times = ds["time"].values

In [ ]:
kwargs = dict(
    n_coarsen=3,
    base_s_thresh=0.55,
    alignment_thresh=0.6,
    int_thresh_factor=0.6,
    hole_size=6,
    smooth_func=partial(gaussian_smooth_func, sigma_lon=2, sigma_lat=0.8),
)
jets, ph_jets, props, props_full = do_everything(ds, path, **kwargs)

In [ ]:
jets_ = ph_jets.filter(
    pl.col("time").dt.year() == 2019,
    pl.col("time").dt.month() == 8,
)

In [ ]:
args = ["all", None, -100, 60, 0, 88]
huh = ("APVO", "thetalev", "APVO", {})
rename, levtype, name, kwargs = huh
da_ = open_da("ERA5", levtype, name, "6H", *args, **kwargs).rename(rename)

In [ ]:
da__ = da_.isel(time=(da_.time.dt.year == 2019) & (da_.time.dt.month == 8)).sel(lev=slice(320, 350)).any("lev").load()

In [ ]:
bias_correction = pl.read_parquet(path.joinpath("bias_correct.parquet"))

In [ ]:
with_interp = create_jet_relative_dataset(jets_, da__, bias_correction=bias_correction)

In [ ]:
with_interp.filter(pl.col("jet ID") == 1).unique("time")

In [ ]:
args = ["all", None, -100, 60, 0, 88]

to_do = (
    ("t2m", "surf", "t2m", {}),
    ("tp", "surf", "tp", {}),
    ("pv330", "thetalev", "PV330", {}),
    ("pv350", "thetalev", "PV350", {}),
    ("APVO", "thetalev", "APVO", {}),
    ("CPVO", "thetalev", "CPVO", {}),
    ("theta", "surf", ("alot2pvu", "theta"), {}),
    ("EKE250", "plev", ("eddy_stuff", "EKE"), {}),
)

bias_correction = pl.read_parquet(path.joinpath("bias_correct.parquet"))

for huh in to_do:
    rename, levtype, name, kwargs = huh
    ofile = path.joinpath(f"{rename}_relative.parquet")
    if ofile.is_file():
        continue
    da = open_da("ERA5", levtype, name, "6H", *args, **kwargs).rename(rename)
    if rename in ["APVO", "CPVO"]:
        da = da.sel(lev=slice(320, 350)).any("lev")
    interpd = create_jet_relative_dataset(jets, da, bias_correction=bias_correction)
    del da
    interpd.write_parquet(ofile)

In [ ]:
xr.open_zarr("/storage/workspaces/giub_meteo_impacts/ci01/ERA5/plev/eddy_stuff/6H/full.zarr")

In [ ]:
open_da("ERA5", levtype, name, "6H", *args, **kwargs).rename(rename)

In [ ]:
rename

In [ ]:
xys = []
all_jets = jets_coarse_bc
# .filter(pl.col("int") > 1.1e8)

xys = np.array(xys)
fig, axes = plt.subplots(
    3, 4, figsize=(11, 9), constrained_layout=True, sharex="all", sharey="all"
)
axes = axes.ravel()
pair = ["s", "theta", "is_polar"]
cmap = LinearSegmentedColormap.from_list(
    "pinkredpurple", [COLORS[2], COLORS_EXT[8], COLORS[1]]
)
bins = np.linspace(0, 1, 31)
gridsize = 18
for month in range(1, 13):
    ax = axes[month - 1]
    X = extract_features(all_jets, pair, season=month)
    probas = X[pair[2]]
    center_stj = X.filter(pl.col("is_polar") < 0.3).mean()
    center_edj = X.filter(pl.col("is_polar") > 0.7).mean()
    X1D = X["is_polar"]

    im1 = ax.hexbin(X[pair[0]], X[pair[1]], cmap=colormaps.gray_r, gridsize=gridsize)
    im2 = ax.hexbin(
        X[pair[0]], X[pair[1]], C=probas, cmap=colormaps.gray_r, gridsize=gridsize
    )

    plt.draw()

    offsets1 = np.asarray(list(map(tuple, im1.get_offsets())), dtype="f, f")
    offsets2 = np.asarray(list(map(tuple, im2.get_offsets())), dtype="f, f")
    mask12 = np.isin(offsets1, offsets2)
    colors = cmap(im2.get_array())
    colors = rgb_to_hsv(colors[:, :3])
    min_s, max_s = 0.0, 0.9
    min_v, max_v = 0.9, 1.0
    scaling = np.clip(
        im1.get_array()[mask12] / im1.get_array()[mask12].max() * 1.1, 0, 1
    )
    f = lambda x: np.sqrt(x)
    colors[:, 1] = min_s + scaling * (max_s - min_s)
    colors[:, 2] = max_v - scaling * (max_v - min_v)
    colors = hsv_to_rgb(colors)
    im2.set_array(None)
    im2.set_facecolor(colors)
    # im2.set_linewidths(0.2)
    im2.set_linewidths(np.clip(2 - 3 * scaling, 0, 2))
    im2.set_edgecolor("white")
    im2 = ax.add_collection(im2)
    if month > 8:
        ax.set_xlabel(pair[0])
    if month % 4 == 0:
        ax.set_ylabel(pair[1])
    if pair[0] in ["lev", "theta"]:
        ax.invert_xaxis()
    elif pair[1] in ["lev", "theta"]:
        ax.invert_yaxis()

    ax.set_title(MONTH_NAMES[month - 1])
    ax.scatter(
        *pl.concat([center_stj, center_edj])[pair[:2]].to_numpy().T,
        facecolor="black",
        edgecolor="white",
        marker="X",
        linewidths=1,
        s=45,
    )
    iax = ax.inset_axes([0.65, 0.14, 0.5, 0.5])
    X1D = np.clip(X1D, 0, 1)
    iax.hist(X1D, bins=bins, alpha=0.5, color="black")
    iax.hist(X1D[probas > 0.5], bins=bins, alpha=1.0, color=COLORS[1])
    iax.hist(X1D[probas < 0.5], bins=bins, alpha=1.0, color=COLORS[2])
    iax.set_xticks([0, 0.5, 1])
    iax.set_yticks([])
    iax.spines[["left", "right", "top"]].set_visible(False)
    iax.set_facecolor((1.0, 1.0, 1.0, 0.0))
    plt.draw()
# fig.savefig(f"{FIGURES}/FeatureBased/gmix_demo.pdf")

# 2023-24 wind data

In [ ]:
basepath = Path(DATADIR, "ERA5/plev/high_wind/6H")
for year in [2024]:
    to_concat = []
    yearstr = str(year).zfill(4)
    for month in range(1, 13):
        monthstr = str(month).zfill(2)
        ds = standardize(
            xr.open_dataset(basepath.joinpath(f"{yearstr}{monthstr}_raw.nc"))
        )
        ds["t"] = standardize(
            xr.open_dataarray(basepath.joinpath(f"{yearstr}{monthstr}_raw_t.nc"))
        )
        opath = basepath.joinpath()
        ds["s"] = np.sqrt(ds["u"] ** 2 + ds["v"] ** 2)
        ds = flatten_by(ds, "s")
        ds["theta"] = ds["t"] * (1000 / ds["lev"]) ** KAPPA
        ds = ds.drop_vars("t")
        to_concat.append(ds)
    to_concat = xr.concat(to_concat, dim="time")
    to_concat.to_netcdf(basepath.joinpath(f"{yearstr}.nc"))

In [ ]:
ds.drop_vars("t")

In [ ]:
xr.open_dataset(basepath.joinpath("2022.nc"))

# new pvs

In [ ]:
import geopandas as gpd
from jetutils.definitions import *
from tqdm import tqdm
from wavebreaking import to_xarray as to_xarray_orig

all_events = {}
basepath = Path(DATADIR, "ERA5/RWB_index/pv")
levs = list(range(310, 365, 5))
for level in levs:
    events = gpd.read_parquet(
        basepath.joinpath(f"era5_pv_streamers_{level}K_1959-2022.parquet")
    )
    stratospheric = events[events.mean_var >= 2]
    tropospheric = events[events.mean_var < 2]

    all_events[f"SAPVS_{level}"] = stratospheric[stratospheric.intensity >= 0]
    all_events[f"TAPVS_{level}"] = tropospheric[tropospheric.intensity >= 0]
    all_events[f"SCPVS_{level}"] = stratospheric[stratospheric.intensity < 0]
    all_events[f"TCPVS_{level}"] = tropospheric[tropospheric.intensity < 0]

    events = gpd.read_parquet(
        basepath.joinpath(f"era5_pv_overturnings_{level}K_1959-2022.parquet")
    )

    all_events[f"APVO_{level}"] = events[events.orientation == "anticyclonic"]
    all_events[f"CPVO_{level}"] = events[events.orientation == "cyclonic"]


varname = "flag"
dtype = {
    "flag": np.uint32,
    "intensity": np.float32,
    "mean_var": np.float32,
    "event_area": np.float32,
}[varname]
coords = {
    "time": TIMERANGE,
    "lat": np.arange(0, 90.5, 1),
    "lon": np.arange(-100, 60.5, 1),
}
shape = [len(co) for co in coords.values()]
dummy_da = xr.DataArray(np.zeros(shape, dtype=dtype), coords=coords)

basepath = Path(DATADIR, "ERA5/thetalev")
sublevels = list(range(310, 355, 5))
for name in ["SAPVS", "TAPVS", "SCPVS", "TCPVS", "APVO", "CPVO"]:
    da = []
    for lev in tqdm(levs):
        da.append(to_xarray_orig(dummy_da, all_events[f"{name}_{lev}"], flag=varname))
    da = xr.concat(da, dim="lev").assign_coords(lev=levs)
    basepath.joinpath(f"{name}/6H").mkdir(parents=True, exist_ok=True)
    da.to_netcdf(basepath.joinpath(f"{name}/6H/full.nc"))
    da = da.sel(lev=sublevels).any("lev").resample(time="1D").any().astype(np.uint8)
    basepath.joinpath(f"{name}/dailyany").mkdir(parents=True, exist_ok=True)
    da.to_netcdf(basepath.joinpath(f"{name}/dailyany/full.nc"))

In [ ]:
xr.open_dataarray(
    "/storage/workspaces/giub_meteo_impacts/ci01/ERA5/thetalev/SAPVS/6H/full.nc"
)

In [ ]:
import shutil
from itertools import product

streamers_subtypes = [
    "_".join(both)
    for both in product(["stratospheric", "tropospheric"], ["anticyclonic", "cyclonic"])
]
types = {"overturnings": ["anticyclonic", "cyclonic"], "streamers": streamers_subtypes}
basepath = Path(DATADIR, "ERA5/thetalev")
for type_, subtypes in types.items():
    for subtype in subtypes:
        if "_" in subtype:
            shorthand_1 = "".join([sub[0].upper() for sub in subtype.split("_")])
            shorthand_2 = subtype.split("_")
            shorthand_2 = shorthand_2[1][:4] + "_" + shorthand_2[0].rstrip("spheric")
        else:
            shorthand_1 = subtype[0].upper()
            shorthand_2 = subtype[:4]
        file_stem = f"{type_}_{shorthand_2}"
        shorthand = shorthand_1 + "PV" + type_[0].upper()
        this_path = basepath.joinpath(shorthand)
        this_path.mkdir(exist_ok=True)
        for freq in ["6H", "dailyany"]:
            file_spec = "" if freq == "6H" else f"_{freq}"
            file_stem_ = f"{type_}_{shorthand_2}_natl{file_spec}.nc"
            this_path_ = this_path.joinpath(freq)
            this_path_.mkdir(exist_ok=True)
            source = Path(DATADIR, "ERA5", "RWB_index", "pv").joinpath(f"{file_stem_}")
            dest = this_path_.joinpath("full.nc")
            print(source, dest)
            shutil.copy(source, dest)
        # print(this_path, shorthand_2)

In [ ]:
basepath = Path(DATADIR, "ERA5/RWB_index/pv")
files_to_treat = basepath.glob("*.nc")
levels = list(range(310, 355, 5))
for f in tqdm(files_to_treat):
    da = xr.open_dataarray(f)
    da = da.sel(lev=levels).any("lev").resample(time="1D").any().astype(np.uint8)
    da.to_netcdf(f.parent.joinpath(f"{f.stem}_dailyany.nc"))

In [ ]:
from jetutils.data import *

basepath = Path(DATADIR, "ERA5/RWB_index/pv")
files_to_treat = basepath.glob("*dailyany.nc")
for f in tqdm(files_to_treat):
    da = xr.open_dataarray(f)
    clim = smooth(compute_clim(da, "dayofyear"), {"dayofyear": ("win", 15)}).astype(
        np.float32
    )
    anom = da.astype(np.float32).groupby("time.dayofyear") - clim
    clim.astype(np.float32).to_netcdf(f.parent.joinpath(f"{f.stem}_clim.nc"))
    anom.astype(np.float32).to_netcdf(f.parent.joinpath(f"{f.stem}_anom.nc"))

In [ ]:
for rwb_type in ["APVO", "CPVO", "SAPVS", "SCPVS", "TAPVS", "TCPVS"]:
    dh = DataHandler.from_specs(
        "ERA5", "thetalev", "APVO", "dailyany", "all", None, -80, 40, 15, 80
    )
    da = compute(dh.da)

# CESM clims

In [ ]:
da_tp = xr.open_zarr(
    "/storage/workspaces/giub_meteo_impacts/ci01/CESM2/PRECL/past.zarr"
)

clim = da_tp.groupby("time.dayofyear").mean()
clim = smooth(clim, {"dayofyear": ("win", 15)})
clim = compute(clim, progress_flag=True)
clim.to_zarr("/storage/workspaces/giub_meteo_impacts/ci01/CESM2/PRECL/past_clim.zarr")

In [ ]:
anom = da_tp.groupby("time.dayofyear") - clim
anom = compute(anom, progress_flag=True)
anom.to_zarr("/storage/workspaces/giub_meteo_impacts/ci01/CESM2/PRECL/past_anom.zarr")

In [ ]:
da_T = xr.open_zarr("/storage/workspaces/giub_meteo_impacts/ci01/CESM2/TS/past.zarr")

clim = da_tp.groupby("time.dayofyear").mean()
clim = smooth(clim, {"dayofyear": ("win", 15)})
clim = compute(clim, progress_flag=True)
clim.to_zarr("/storage/workspaces/giub_meteo_impacts/ci01/CESM2/PRECL/past_clim.zarr")

In [ ]:
# da_tp = xr.open_zarr("/storage/workspaces/giub_meteo_impacts/ci01/CESM2/PRECL/future.zarr")

# clim = da_tp.groupby("time.dayofyear").mean()
# clim = smooth(clim, {'dayofyear': ('win', 15)})
# clim = compute(clim, progress_flag=True)
# clim.to_zarr("/storage/workspaces/giub_meteo_impacts/ci01/CESM2/PRECL/future_clim.zarr")

# create jet relative climatologies

In [ ]:
all_times = (
    pl.datetime_range(
        start=pl.datetime(1959, 1, 1),
        end=pl.datetime(2023, 1, 1),
        closed="left",
        interval="6h",
        eager=True,
        time_unit="ms",
    )
    .rename("time")
    .to_frame()
)
summer_filter = all_times.filter(pl.col("time").dt.month().is_in([6, 7, 8, 9])).filter(
    pl.col("time").dt.ordinal_day() > 166
)
summer = summer_filter["time"]
summer_daily = summer.filter(summer.dt.hour() == 0)
big_summer = all_times.filter(pl.col("time").dt.month().is_in([6, 7, 8, 9]))
big_summer_daily = big_summer.filter(big_summer["time"].dt.hour() == 0)

dh = DataHandler(
    "/storage/workspaces/giub_meteo_impacts/ci01/ERA5/plev/high_wind/6H/results/8"
)
exp = JetFindingExperiment(dh)
ds = exp.ds
all_jets_one_df = exp.find_jets(force=False, base_s_thresh=0.55, hole_size=6)
all_jets_one_df = exp.categorize_jets(
    None, ["s", "theta"], force=False, n_init=10, init_params="k-means++", mode="week"
).cast({"time": pl.Datetime("ms")})
props_uncat = exp.props_as_df(False).cast({"time": pl.Datetime("ms")})

props_as_df = average_jet_categories(props_uncat, polar_cutoff=0.5)

props_summer = summer_filter.join(props_as_df, on="time")
phat_filter = (pl.col("is_polar") < 0.5) | (
    (pl.col("is_polar") > 0.5) & (pl.col("int") > 1.3e8)
)

phat_jets = all_jets_one_df.filter(
    (pl.col("is_polar").mean().over(["time", "jet ID"]) < 0.5)
    | (
        (pl.col("is_polar").mean().over(["time", "jet ID"]) > 0.5)
        & (pl.col("int").mode().first().over(["time", "jet ID"]) > 1.3e8)
    )
)
phat_jets_catd = phat_jets.with_columns(
    **{
        "jet ID": (pl.col("is_polar").mean().over(["time", "jet ID"]) > 0.5).cast(
            pl.UInt32()
        )
    }
)
phat_props = props_uncat.filter(phat_filter)
phat_props_catd = average_jet_categories(phat_props, polar_cutoff=0.5)

phat_props_catd = phat_props_catd.join(
    phat_props_catd.rolling("time", period="2d", group_by="jet").agg(
        **{
            f"{col}_var": pl.col(col).var()
            for col in ["mean_lon", "mean_lat", "mean_s", "s_star"]
        }
    ),
    on=["time", "jet"],
)

phat_props_catd_summer = summer_filter.join(phat_props_catd, on="time")

cross_catd_ofile = exp.path.joinpath("cross_catd.parquet")
if cross_catd_ofile.is_file():
    cross_catd = pl.read_parquet(cross_catd_ofile)
else:
    cross_catd = track_jets(phat_jets_catd)
    cross_catd.write_parquet(cross_catd_ofile)

pers = pers_from_cross_catd(cross_catd)
spells_list = spells_from_cross_catd_simple(
    cross_catd,
    season=summer,
    q_STJ=0.705,
    q_EDJ=0.836,
    minlen=datetime.timedelta(days=5),
    smooth=datetime.timedelta(hours=24),
    fill_holes=datetime.timedelta(hours=18),
)

daily_spells_list = {
    a: make_daily(b, "spell", ["len", "spell_of"]) for a, b in spells_list.items()
}

In [ ]:
def compute_emf_2d_conv(ds):
    lon, lat = ds["lon"].values, ds["lat"].values
    xlon, ylat = np.meshgrid(lon, lat)

    _, dlonx = np.gradient(xlon)
    dlaty, _ = np.gradient(ylat)

    dx = RADIUS * np.radians(dlaty) * degcos(ylat)
    dy = RADIUS * np.radians(dlaty)

    e1 = 0.5 * (ds["vp"] ** 2 - ds["up"] ** 2)
    e2 = -ds["up"] * ds["vp"]
    de1dx = ds["up"].copy(data=np.gradient(e1, axis=2)) / dx[None, :, :]
    de2dy = ds["up"].copy(data=np.gradient(e2, axis=1)) / dy[None, :, :]
    return de1dx + de2dy

In [ ]:
args = ["all", None, -100, 60, 0, 90]

da_T = open_da("ERA5", "surf", "t2m", "dailymean", *args)
da_T = compute(da_T)
create_jet_relative_dataset(phat_jets_catd, exp.path, da_T, suffix="_phat_catd")
del da_T

da_tp = open_da("ERA5", "surf", "tp", "dailysum", *args)
da_tp = compute(da_tp)
create_jet_relative_dataset(phat_jets_catd, exp.path, da_tp, suffix="_phat_catd")
del da_tp

da_pv = open_da("ERA5", "thetalev", "PV330", "dailymean", *args)
da_pv = compute(da_pv).rename("PV330")
create_jet_relative_dataset(phat_jets_catd, exp.path, da_pv, suffix="_phat_catd")
del da_pv

da_pv = open_da("ERA5", "thetalev", "PV350", "dailymean", *args)
da_pv = compute(da_pv)
create_jet_relative_dataset(phat_jets_catd, exp.path, da_pv, suffix="_phat_catd")
del da_pv

varnames_rwb = ["APVO", "CPVO"]
for var in varnames_rwb:
    da_rwb = open_da("ERA5", "thetalev", var, "dailyany", *args)
    da_rwb = compute(da_rwb)
    create_jet_relative_dataset(phat_jets_catd, exp.path, da_rwb, suffix="_phat_catd")
    del da_rwb

da_theta2pvu = open_da("ERA5", "surf", "alot2pvu", "dailymean", *args)
da_theta2pvu = compute(da_theta2pvu)
create_jet_relative_dataset(phat_jets_catd, exp.path, da_theta2pvu, suffix="_phat_catd")
del da_theta2pvu

ds = xr.open_zarr(
    "/storage/workspaces/giub_meteo_impacts/ci01/ERA5/plev/uv/6H/results/Eddy_uv_natl_6days.zarr"
)

EKE = ds["EKE"].sel(lev=250).rename("EKE250")
EKE = EKE.resample(time="1d").mean()
EKE = compute(EKE, progress_flag=True)
create_jet_relative_dataset(phat_jets_catd, exp.path, EKE, suffix="_phat_catd")
del EKE

EMF = ds["EMF"].sel(lev=250).rename("EMF250")
EMF = compute(EMF, progress_flag=True)
EMF = EMF.resample(time="1d").mean()
create_jet_relative_dataset(phat_jets_catd, exp.path, EMF, suffix="_phat_catd")
del EMF

up = ds["up"].sel(lev=250).rename("up250")
up = up.resample(time="1d").mean()
up = compute(up, progress_flag=True)
create_jet_relative_dataset(phat_jets_catd, exp.path, up, suffix="_phat_catd")
del up

vp = ds["vp"].sel(lev=250).rename("vp250")
vp = compute(vp, progress_flag=True)
vp = vp.resample(time="1d").mean()
create_jet_relative_dataset(phat_jets_catd, exp.path, vp, suffix="_phat_catd")
del vp

In [ ]:
args = ["all", None, -100, 60, 0, 90]

# da_T = open_da("ERA5", "surf", "t2m", "dailymean", *args)
# da_T = compute(da_T)
# create_jet_relative_dataset(phat_jets_catd, exp.path, da_T, suffix="_phat_catd")
# del da_T

# da_tp = open_da("ERA5", "surf", "tp", "dailysum", *args)
# da_tp = compute(da_tp)
# create_jet_relative_dataset(phat_jets_catd, exp.path, da_tp, suffix="_phat_catd")
# del da_tp

# da_pv = open_da("ERA5", "thetalev", "PV330", "dailymean", *args)
# da_pv = compute(da_pv).rename("PV330")
# create_jet_relative_dataset(phat_jets_catd, exp.path, da_pv, suffix="_phat_catd")
# del da_pv

# da_pv = open_da("ERA5", "thetalev", "PV350", "dailymean", *args)
# da_pv = compute(da_pv)
# create_jet_relative_dataset(phat_jets_catd, exp.path, da_pv, suffix="_phat_catd")
# del da_pv

# varnames_rwb = ["APVO", "CPVO"]
# for var in varnames_rwb:
#     da_rwb = open_da("ERA5", "thetalev", var, "dailyany", *args)
#     da_rwb = compute(da_rwb)
#     create_jet_relative_dataset(phat_jets_catd, exp.path, da_rwb, suffix="_phat_catd")
#     del da_rwb

# da_theta2pvu = open_da("ERA5", "surf", ("alot2pvu", "theta"), "dailymean", *args)
# da_theta2pvu = compute(da_theta2pvu)
# create_jet_relative_dataset(phat_jets_catd, exp.path, da_theta2pvu, suffix="_phat_catd")
# del da_theta2pvu

ds = xr.open_zarr(
    "/storage/workspaces/giub_meteo_impacts/ci01/ERA5/plev/uv/6H/results/Eddy_uv_natl_10days.zarr"
)

# up = ds["up"].sel(lev=250).rename("up250")
# up = up.resample(time="1d").mean()
# up = compute(up, progress_flag=True)
# create_jet_relative_dataset(phat_jets_catd, exp.path, up, suffix="_phat_catd")
# del up

# vp = ds["vp"].sel(lev=250).rename("vp250")
# vp = compute(vp, progress_flag=True)
# vp = vp.resample(time="1d").mean()
# create_jet_relative_dataset(phat_jets_catd, exp.path, vp, suffix="_phat_catd")
# del vp

# EMFconv = compute_emf_2d_conv(ds.sel(lev=250)).rename("EMFconv250")
# EMFconv = compute(EMFconv, progress_flag=True)
# EMFconv = EMFconv.resample(time="1d").mean()
# create_jet_relative_dataset(phat_jets_catd, exp.path, EMFconv, suffix="_phat_catd")
# del EMFconv

EKE = (ds.sel(lev=250)["up"] ** 2 + ds.sel(lev=250)["vp"] ** 2) * 0.5
EKE = compute(EKE, progress_flag=True).rename("EKE250")
EKE = EKE.resample(time="1d").mean()
create_jet_relative_dataset(phat_jets_catd, exp.path, EKE, suffix="_phat_catd")
del EKE

In [ ]:
from jetutils.data import compute_all_dailymeans

compute_all_dailymeans("ERA5", "surf", "alot2pvu")

In [ ]:
from pathlib import Path

base_path_1 = Path(f"{DATADIR}/ERA5/surf/theta2PVU/6H")
base_path_2 = Path(f"{DATADIR}/ERA5/surf/theta2PVU/dailymean")
# base_path_2.mkdir()
for year in YEARS:
    print(year, end="\r")
    opath_1 = base_path_1.joinpath(f"{year}.nc")
    opath_2 = base_path_2.joinpath(f"{year}.nc")

    if opath_2.is_file():
        continue

    this_pv = standardize(open_dataarray(opath_1))
    this_pv = compute(this_pv, progress_flag=False)
    this_pv = this_pv.resample(time="1d").mean()
    this_pv.to_netcdf(opath_2)

# arco-era5 tests

In [ ]:
ds = xr.open_zarr(
    "gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3",
    chunks=None,
    storage_options=dict(token="anon"),
)
ar_full_37_1h = ds.sel(
    time=slice(ds.attrs["valid_time_start"], ds.attrs["valid_time_stop"])
)

temp_full = (
    ar_full_37_1h["temperature"]
    .sel(
        time=ar_full_37_1h.time.dt.hour % 6 == 0,
        latitude=ar_full_37_1h.latitude >= 0,
        level=200,
    )
    .isel(longitude=slice(None, None, 2), latitude=slice(None, None, 2))
)

temp_full = standardize(temp_full).chunk("auto")

from pathlib import Path

base_path_1 = Path(f"{DATADIR}/ERA5/plev/t200/6H")
base_path_2 = Path(f"{DATADIR}/ERA5/plev/t200/dailymean")
# base_path_1.mkdir(parents=True)
# base_path_2.mkdir(parents=True)
for year in YEARS:
    print(year)
    opath_1 = base_path_1.joinpath(f"{year}.nc")
    opath_2 = base_path_2.joinpath(f"{year}.nc")

    if opath_2.is_file():
        continue
    this_temp = temp_full.sel(time=temp_full.time.dt.year == year)
    this_temp = this_temp.reset_coords("lev", drop=True)
    this_temp = compute(this_temp, progress_flag=True)
    this_temp.to_netcdf(opath_1)

    this_temp = this_temp.resample(time="1d").mean()
    this_temp.to_netcdf(opath_2)

In [ ]:
temp_full

In [ ]:
ds = xr.open_zarr(
    "gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3",
    chunks=None,
    storage_options=dict(token="anon"),
)
ar_full_37_1h = ds.sel(
    time=slice(ds.attrs["valid_time_start"], ds.attrs["valid_time_stop"])
)

temp_full = (
    ar_full_37_1h["temperature"]
    .sel(
        time=ar_full_37_1h.time.dt.hour % 6 == 0,
        latitude=ar_full_37_1h.latitude >= 0,
        level=[175, 200, 225, 250, 300, 350],
    )
    .isel(longitude=slice(None, None, 2), latitude=slice(None, None, 2))
)

temp_full = standardize(temp_full)

orig_path = Path(f"{DATADIR}/ERA5/plev/flat_wind/dailymean")
base_path = Path(f"{DATADIR}/ERA5/plev/flat_wind/dailymean_2")
for year in tqdm(YEARS):
    for month in trange(1, 13, leave=False):
        month_str = str(month).zfill(2)
        opath = base_path.joinpath(f"{year}{month_str}.nc")
        if opath.is_file():
            continue
        ipath = orig_path.joinpath(f"{year}{month_str}.nc")
        ds = xr.open_dataset(ipath)
        this_temp = temp_full.sel(time=ds.time.values, lev=ds["lev"])
        this_temp = this_temp * (1000 / this_temp.lev) ** KAPPA
        this_temp = this_temp.reset_coords("lev", drop=True)
        ds["theta"] = compute(this_temp, progress_flag=True)
        ds.to_netcdf(opath)

# CESM

### new cesm zarrification

In [ ]:
basepath = Path(f"{DATADIR}/CESM2/high_wind/ssp370")
paths = list(basepath.glob("*-*.nc"))
names = [path.stem.split("-") for path in paths]
members = [name[0] for name in names]
years = [name[1] for name in names]
for i, member in enumerate(tqdm(np.unique(members))):
    ds = standardize(xr.open_mfdataset(basepath.joinpath(f"{member}-*.nc").as_posix()))
    ds_temp = standardize(
        xr.open_mfdataset(basepath.joinpath(f"raw/{member}-*.nc").as_posix())
    )
    ds["t"] = (
        ds_temp["t"]
        .assign_coords(time=ds.time.values)
        .sel(lat=ds["lat"].values, lon=ds["lon"].values)
        .sel(lev=ds["lev"])
        .reset_coords("lev", drop=True)
    )
    ds["member"] = ds["member"].astype("<U15")
    ds = ds.expand_dims("member").copy(deep=True)
    for var in ds.data_vars:
        ds[var] = ds[var].astype(np.float32)
    kwargs = {"mode": "w"} if i == 0 else {"mode": "a", "append_dim": "member"}
    ds.to_zarr(basepath.joinpath("ds.zarr"), **kwargs)